In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,PowerTransformer,RobustScaler,OneHotEncoder
from sklearn.metrics import accuracy_score,recall_score,f1_score,precision_score,roc_auc_score,confusion_matrix,classification_report
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from collections import Counter
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from sklearn.utils.class_weight import compute_class_weight
import optuna
from tensorflow.keras.optimizers import RMSprop,SGD,Adam
from tensorflow.keras.callbacks import EarlyStopping

I0000 00:00:1795416165.890132      50 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795416185.226533      50 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
df = pd.read_csv(r"/kaggle/input/datasets/santhoshkumarpuppala/american-bank/american_bankruptcy_dataset.csv")
pd.set_option('display.max_columns', None)

In [5]:
df.rename(columns={
    "X1": "Current_Assets",
    "X2": "Cost_of_Goods_Sold",
    "X3": "Depreciation_and_Amortization",
    "X4": "EBITDA",
    "X5": "Inventory",
    "X6": "Net_Income",
    "X7": "Total_Receivables",
    "X8": "Market_Value",
    "X9": "Net_Sales",
    "X10": "Total_Assets",
    "X11": "Long_Term_Debt",
    "X12": "EBIT",
    "X13": "Gross_Profit",
    "X14": "Current_Liabilities",
    "X15": "Retained_Earnings",
    "X16": "Total_Revenue",
    "X17": "Total_Liabilities",
    "X18": "Operating_Expenses"}, inplace=True)

In [6]:
df.drop(columns=["Total_Liabilities"], inplace=True)

In [7]:
train = df[df["fyear"].between(1999, 2011)]
val = df[df["fyear"].between(2012, 2014)]
test = df[df["fyear"].between(2015, 2018)]

print("Training:", train.shape)
print("Validation:", val.shape)
print("Testing:", test.shape)

Training: (55927, 22)
Validation: (10473, 22)
Testing: (12282, 22)


In [8]:
X_train = train.drop(columns = ['company_name','status_label'])
y_train = train['status_label']
X_val = val.drop(columns = ['company_name','status_label'])
y_val = val['status_label']
X_test = test.drop(columns = ['company_name','status_label'])
y_result = test['status_label']

In [9]:
numerical_pipeline = Pipeline([("Scaler", RobustScaler())])

In [10]:
categorical_pipeline = Pipeline([("Encoder", OneHotEncoder(handle_unknown="ignore",sparse_output=True))])

In [11]:
numerical_features = ["Current_Assets","Cost_of_Goods_Sold","Depreciation_and_Amortization","EBITDA","Inventory","Net_Income","Total_Receivables","Market_Value","Net_Sales","Total_Assets","Long_Term_Debt",
    "EBIT","Gross_Profit","Retained_Earnings","Total_Revenue","Current_Liabilities","Operating_Expenses","fyear"]

categorical_features = ["MajorGroup",'Division']

In [12]:
df['Division'].value_counts()

Division
D    38384
I    17148
E     7203
G     5695
B     4181
F     2939
H     2060
C      721
A      338
J       13
Name: count, dtype: int64

In [13]:
preprocessor = ColumnTransformer([("Numerical", numerical_pipeline, numerical_features),("Categorical", categorical_pipeline, categorical_features)])

In [14]:
classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

print("Class Weights:", class_weights)

Class Weights: {'alive': np.float64(0.5431387782849374), 'failed': np.float64(6.295249887438091)}


In [15]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

In [16]:
label_map = {'alive': 0, 'failed': 1}
y_train = train['status_label'].map(label_map)
y_val = val['status_label'].map(label_map)
y_result = test['status_label'].map(label_map)

In [25]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),

    Dense(32, activation='relu'),
    Dropout(0.2),

    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1795415989.913467      49 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [28]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [29]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    class_weight=class_weights,
    verbose=1
)

NameError: name 'y_test' is not defined

In [73]:
y_pred_prob = model.predict(X_test)

y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

492/492 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [74]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))
print()

              precision    recall  f1-score   support

           0       0.97      0.75      0.85     14693
           1       0.16      0.69      0.27      1044

    accuracy                           0.75     15737
   macro avg       0.57      0.72      0.56     15737
weighted avg       0.92      0.75      0.81     15737



In [17]:

def create_objective(metric_name):

    def objective(trial):

        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(21)

        # -------------------------------
        # 1. Hyperparameter search space
        # -------------------------------

        n_layers = trial.suggest_int(
            "n_layers", 1, 4
        )

        learning_rate = trial.suggest_float(
            "learning_rate",
            0.0001,
            0.01,
            log=True
        )

        batch_size = trial.suggest_categorical(
            "batch_size",
            [32, 64, 128, 256]
        )

        optimizer_name = trial.suggest_categorical(
            "optimizer",
            ["adam", "rmsprop", "sgd"]
        )

        # Threshold is selected using validation data
        threshold = trial.suggest_float(
            "threshold",
            0.05,
            0.95,
            step=0.01
        )

        # -------------------------------
        # 2. Build ANN
        # -------------------------------

        model = Sequential()

        model.add(
            Input(shape=(X_train_processed.shape[1],))
        )

        for i in range(n_layers):

            units = trial.suggest_categorical(
                f"units_layer_{i+1}",
                [32, 64, 128, 256]
            )

            dropout_rate = trial.suggest_float(
                f"dropout_layer_{i+1}",
                0.0,
                0.4,
                step=0.1
            )

            activation = trial.suggest_categorical(
                f"activation_layer_{i+1}",
                ["relu", "tanh"]
            )

            model.add(
                Dense(
                    units=units,
                    activation=activation
                )
            )

            if dropout_rate > 0:

                model.add(
                    Dropout(dropout_rate)
                )

        model.add(
            Dense(1, activation="sigmoid")
        )

        # -------------------------------
        # 3. Optimizer selection
        # -------------------------------

        if optimizer_name == "adam":

            optimizer = Adam(
                learning_rate=learning_rate
            )

        elif optimizer_name == "rmsprop":

            optimizer = RMSprop(
                learning_rate=learning_rate
            )

        else:

            optimizer = SGD(
                learning_rate=learning_rate
            )

        # -------------------------------
        # 4. Compile
        # -------------------------------

        model.compile(
            optimizer=optimizer,
            loss="binary_crossentropy",
            metrics=[
                tf.keras.metrics.AUC(
                    curve="PR",
                    name="pr_auc"
                )
            ]
        )

        # -------------------------------
        # 5. Early stopping
        # -------------------------------

        early_stop = EarlyStopping(
            monitor="val_pr_auc",
            mode="max",
            patience=10,
            restore_best_weights=True,
            verbose=0
        )

        # -------------------------------
        # 6. Model training
        # -------------------------------

        history = model.fit(
            X_train_processed,
            y_train,

            validation_data=(
                X_val_processed,
                y_val
            ),

            epochs=150,
            batch_size=batch_size,

            class_weight=class_weights,

            callbacks=[early_stop],
            verbose=0
        )

        # -------------------------------
        # 7. Validation predictions
        # -------------------------------

        val_probabilities = model.predict(
            X_val_processed,
            verbose=0
        ).ravel()

        val_predictions = (
            val_probabilities >= threshold
        ).astype(int)

        precision = precision_score(
            y_val,
            val_predictions,
            zero_division=0
        )

        recall = recall_score(
            y_val,
            val_predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_val,
            val_predictions,
            zero_division=0
        )

        # -------------------------------
        # 8. Save trial information
        # -------------------------------

        trial.set_user_attr(
            "precision", float(precision)
        )

        trial.set_user_attr(
            "recall", float(recall)
        )

        trial.set_user_attr(
            "f1_score", float(f1)
        )

        trial.set_user_attr(
            "threshold", float(threshold)
        )

        trial.set_user_attr(
            "best_pr_auc",
            float(max(history.history["val_pr_auc"]))
        )

        # -------------------------------
        # 9. Select optimization objective
        # -------------------------------

        if metric_name == "precision":

            # Require at least 30% recall
            if recall < 0.30:
                return 0.0

            return float(precision)

        elif metric_name == "recall":

            # Require at least 10% precision
            if precision < 0.10:
                return 0.0

            return float(recall)

        elif metric_name == "f1":

            return float(f1)

    return objective


In [18]:
study_recall = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=21)
)

study_recall.optimize(
    create_objective("recall"),
    n_trials=50,
    n_jobs=1,
    show_progress_bar=True
)

[I 2026-11-23 06:43:11,062] A new study created in memory with name: no-name-96d68d40-98db-4d78-bfdb-aca8872392c2


  0%|          | 0/50 [00:00<?, ?it/s]

E0000 00:00:1795416191.409046      50 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


[I 2026-11-23 06:45:46,264] Trial 0 finished with value: 0.0 and parameters: {'n_layers': 1, 'learning_rate': 0.000378633747241908, 'batch_size': 32, 'optimizer': 'rmsprop', 'threshold': 0.5800000000000001, 'units_layer_1': 64, 'dropout_layer_1': 0.2, 'activation_layer_1': 'relu'}. Best is trial 0 with value: 0.0.
[I 2026-11-23 06:47:44,771] Trial 1 finished with value: 0.06517311608961303 and parameters: {'n_layers': 4, 'learning_rate': 0.00330069279804087, 'batch_size': 128, 'optimizer': 'rmsprop', 'threshold': 0.52, 'units_layer_1': 256, 'dropout_layer_1': 0.2, 'activation_layer_1': 'relu', 'units_layer_2': 128, 'dropout_layer_2': 0.30000000000000004, 'activation_layer_2': 'relu', 'units_layer_3': 256, 'dropout_layer_3': 0.2, 'activation_layer_3': 'tanh', 'units_layer_4': 64, 'dropout_layer_4': 0.1, 'activation_layer_4': 'relu'}. Best is trial 1 with value: 0.06517311608961303.
[I 2026-11-23 06:51:25,142] Trial 2 finished with value: 0.006109979633401222 and parameters: {'n_layers':

In [19]:
print("Best Recall:", study_recall.best_value)

print("\nBest Parameters:")

for key, value in study_recall.best_trial.params.items():
    print(f"{key}: {value}")

print("\nValidation Precision:",
      study_recall.best_trial.user_attrs["precision"])

print("Validation F1:",
      study_recall.best_trial.user_attrs["f1_score"])

Best Recall: 0.7841140529531568

Best Parameters:
n_layers: 3
learning_rate: 0.0022833657973117443
batch_size: 128
optimizer: adam
threshold: 0.060000000000000005
units_layer_1: 128
dropout_layer_1: 0.30000000000000004
activation_layer_1: relu
units_layer_2: 256
dropout_layer_2: 0.30000000000000004
activation_layer_2: relu
units_layer_3: 128
dropout_layer_3: 0.30000000000000004
activation_layer_3: relu

Validation Precision: 0.10187880391637999
Validation F1: 0.18032786885245902
